In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from fastapi import FastAPI,UploadFile
from tensorflow.keras.models import load_model
import numpy as np
import cv2
from pyngrok import ngrok
import uvicorn
import nest_asyncio

app=FastAPI()


model=load_model("/content/drive/MyDrive/Intel_Model.keras")

CLASS_NAMES = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']


@app.get('/')
def home():
  return {"message":"This is my Image Classifciation Project Api"}


@app.post('/predict')
async def prediction_func(image:UploadFile):
  # first read the image and get bytes like "xcdqwdbq...."
  image= await image.read()
  # now bytes convert into numpy array
  image=np.frombuffer(image,np.uint8)
  # now decode this 1-d array using cv2
  image=cv2.imdecode(image,cv2.IMREAD_COLOR)
  # got the 2d image now do resize
  image=cv2.resize(image,(224,224))
  # now do normalization
  image_normalize=image/255.0
  # now add batch dimension
  image_ready=np.expand_dims(image_normalize,axis=0)

  # now do prediction on image
  prediction=model.predict(image_ready)
  # get max probality from prediction
  max_prob_pred=np.argmax(prediction)
  main_prediction=int(max_prob_pred)
  return {"prediction":f"{CLASS_NAMES[main_prediction]}",
      "confidence": round(float(prediction[0][max_prob_pred]) * 100, 2)
}



# ngrok setup
nest_asyncio.apply()

public_url = ngrok.connect(8000)
print(f" Public URL: {public_url}")

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()


 Public URL: NgrokTunnel: "https://344d-35-194-180-146.ngrok-free.app" -> "http://localhost:8000"


INFO:     Started server process [23328]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
INFO:     121.46.65.46:0 - "POST /predict HTTP/1.1" 200 OK
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step
INFO:     121.46.65.46:0 - "POST /predict HTTP/1.1" 200 OK
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step
INFO:     121.46.65.46:0 - "POST /predict HTTP/1.1" 200 OK
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step
INFO:     121.46.65.46:0 - "POST /predict HTTP/1.1" 200 OK
